# 제12장 다기준 의사결정 (Multi-Criteria Decision Making)

## 학습 목표
1. 인지 편향과 구조화된 의사결정의 필요성 이해
2. 가중 점수법(Weighted Scoring)으로 대안 평가
3. AHP(Analytic Hierarchy Process)로 체계적 가중치 도출
4. 의사결정 나무와 정보의 가치(EVPI, EVII) 계산
5. 민감도 분석으로 의사결정 강건성 확보

## 강의 구조
| 파트 | 시간 | 내용 |
|------|------|------|
| Part 1 | | 의사결정의 본질과 인지 편향 |
| Part 2 | | 가중 점수법과 AHP |
| 휴식 | | - |
| Part 3 | | 불확실성하 의사결정과 정보의 가치 |
| Part 4 | | 실습: 가중 점수법과 AHP |
| Part 5 | | 실습: 의사결정 나무와 EVPI |

In [ ]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")

## Part 1: 의사결정의 본질과 인지 편향

### 합리적 의사결정의 가정
경제학의 전통적 모델은 **완전한 정보와 무한한 인지능력**을 가진 의사결정자를 가정합니다.
그러나 현실에서는:
- 정보가 불완전하고 비용이 높음
- 인지적 한계로 인해 휴리스틱(Heuristic)에 의존
- 감정과 편향이 판단에 영향

### Herbert Simon의 "제한된 합리성" (Bounded Rationality)
> **우리는 최적(Optimize)하기보다는 만족(Satisfice)한다**

인간은 완벽한 의사결정 대신 "충분히 좋은" 대안을 찾으려 합니다.

### 주요 인지 편향

| 편향 | 정의 | 기획에서의 영향 | 대응책 |
|------|------|-----------------|--------|
| **확증편향** (Confirmation Bias) | 자신의 믿음을 지지하는 정보만 수집 | 사전 조사 무시, 반대 의견 폄하 | Red Team, Devil's Advocate |
| **앵커링 편향** (Anchoring Bias) | 처음 제시된 수치에 과도하게 의존 | 초기 예산/타겟에 고착 | 독립적 추정, 다중 기준점 |
| **과신편향** (Overconfidence Bias) | 자신의 능력/정보를 과대평가 | 위험 과소평가, 타임라인 단축 | 사전 분석, 외부 벤치마킹 |
| **손실회피** (Loss Aversion) | 같은 크기의 이득보다 손실을 2배 이상 두려워함 | 안전한 기존사업 선호, 혁신 회피 | Expected Value 분석, 포트폴리오 접근 |
| **가용성편향** (Availability Bias) | 최근/기억하기 쉬운 사건에 과도한 가중치 | 한두 사례로 시장 판단, 거품 형성 | 데이터 기반 통계, 롤링 분석 |
| **군집편향** (Herd Behavior) | 합리적 판단과 무관하게 다수를 따라감 | 경쟁사 모방, 유행 추종 | 독립적 분석, 포지셔닝 |

### 집단의사결정 병폐: 집단사고 (Groupthink)

**Irving Janis 정의**: 응집력 높은 집단이 다양한 관점을 무시하고 단일 입장으로 수렴하는 현상

**증상**:
- 대안의 철저한 검토 부재
- 반대 의견에 대한 압력
- 당신 집단의 능력 과대평가
- 위험에 대한 무시

**방지법**:
1. **Premortem** (Gary Klein): "프로젝트가 실패했다고 가정하고 원인을 찾아보세요"
2. **Red Team**: 의도적으로 다른 관점에서 비판
3. **Delphi Method**: 익명의 다회차 의견 수렴
4. **외부 전문가 초청**: 신선한 관점 도입

### 구조화된 의사결정의 필요성

직관(Intuition)을 완전히 배제할 수는 없으나, **구조와 투명성**이 필요:
- **다기준 의사결정(MCDM)**: 여러 목표를 동시에 고려하는 체계적 접근법
- **정량적 분석**: 감정적 편향 감소
- **추적 가능성**: 의사결정 근거의 명문화 → 학습 가능

**결론**: 데이터 + 구조 + 전문가 판단 = 더 나은 의사결정

### 📝 이론 과제 12-1 (15분)

#### 과제: 인지 편향과 의사결정 개선

**질문:**
1. 확증편향(Confirmation Bias)이 신사업 기획 과정에서 어떻게 나타날 수 있는지 구체적인 사례와 함께 설명하세요. (3-4문장)

2. Gary Klein의 Premortem 기법을 설명하고, 이것이 집단사고를 어떻게 방지하는지 설명하세요. (2-3문장)

3. 당신이 속한 조직에서 경험한 인지 편향의 사례를 설명하세요. (실제 사례 또는 가상의 경우)

**조사 키워드:**
- "confirmation bias strategic planning"
- "premortem technique Gary Klein"
- "groupthink prevention Delphi method"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 12-1 제출란

**학번:** ___________
**이름:** ___________

#### 1. 확증편향이 신사업 기획에서 나타나는 방식

(여기에 작성)

#### 2. Premortem 기법과 집단사고 방지

(여기에 작성)

#### 3. 조직에서 경험한 인지 편향 사례

(여기에 작성)

## Part 2: 가중 점수법과 AHP

### 다기준 의사결정(MCDM)의 기본 구조

```
                           Goal (목표)
                             |
              _______________+_______________
             |       |        |       |      |
          Criterion 1  Criterion 2 ... Criterion n
             |       |        |       |      |
          ____________+____________
         |       |        |       |      |
      Alt. A  Alt. B   Alt. C ... Alt. M
```

**핵심 요소:**
- **목표(Goal)**: 최상위 의사결정 목표 (예: 최적 사업 전략)
- **기준(Criteria)**: 대안 평가를 위한 측정 기준 (다양한 차원)
- **가중치(Weights)**: 각 기준의 상대적 중요도
- **대안(Alternatives)**: 비교 대상 선택지

### 가중 점수법 (Weighted Scoring Method)

**4단계 프로세스:**

**Step 1: 기준 정의**
- 기획 목표와 관련된 평가 기준 나열
- 상호 독립적이고 포괄적이어야 함

**Step 2: 가중치 부여**
- 각 기준의 상대적 중요도 정량화
- 방법: 직접 할당, 쌍대 비교, 비율 스케일링

**Step 3: 대안별 점수 매기기**
- 1~10점 또는 0~100점 척도 사용
- 기준별로 대안의 성과 평가

**Step 4: 가중 점수 계산**
$$
W_i = \sum_{j=1}^{n} w_j \times s_{ij}
$$

여기서:
- $W_i$ = 대안 i의 총 가중 점수
- $w_j$ = 기준 j의 가중치
- $s_{ij}$ = 대안 i가 기준 j에서 받은 점수

**장점**: 계산 간단, 직관적, 이해하기 쉬움
**단점**: 가중치 결정의 자의성, 기준 간 독립성 미흡

### AHP (Analytic Hierarchy Process)

**Saaty(1977) 개발** — 가중치를 더 체계적으로 도출

**핵심 아이디어**: 쌍대 비교(Pairwise Comparison)
- 두 기준을 직접 비교하는 것이 여러 기준을 동시에 비교하는 것보다 쉬움
- 비교 결과로부터 상대적 가중치를 수학적으로 도출

**Saaty 9점 척도:**
| 평가 | 의미 | 예시 |
|------|------|------|
| 1 | 동등함 | 기준 A와 B가 똑같이 중요 |
| 3 | 약간 중요 | 기준 A가 B보다 약간 중요 |
| 5 | 명확히 중요 | 기준 A가 B보다 명확히 중요 |
| 7 | 매우 중요 | 기준 A가 B보다 매우 중요 |
| 9 | 절대적으로 중요 | 기준 A가 B보다 절대적으로 중요 |
| 2,4,6,8 | 중간값 | |

**Step 1: 쌍대 비교 행렬 구성**
```
       마케팅   수익성   위험    실행성   시너지
마케팅   1      2      3       4       5
수익성  1/2     1      2       3       4
위험    1/3    1/2     1       2       3
실행성  1/4    1/3    1/2      1       2
시너지  1/5    1/4    1/3     1/2      1
```

**Step 2: 우선순위 벡터 계산 (Column Normalization)**
- 각 열의 합 계산
- 각 원소를 열의 합으로 정규화
- 각 행의 평균 = 우선순위 가중치

**Step 3: 일관성 검증**
- 일관성 지수 (CI) = (λmax - n) / (n - 1)
- 일관성 비율 (CR) = CI / RI (RI: 난수 지수)
- **기준**: CR < 0.10이면 합리적

**장점**: 논리적 엄밀성, 일관성 검증 가능, 전문가 합의 용이
**단점**: 계산이 복잡, 쌍대 비교 필요

### 민감도 분석 (Sensitivity Analysis)

**목적**: 가중치 변화에 따라 대안의 순위가 어떻게 변하는가?

**방법**: 한 번에 하나의 가중치를 변화시키며 최종 점수 추적
- 어떤 기준의 가중치가 가장 큰 영향을 미치는가?
- 순위 역전이 일어나는 가중치는?

**의사결정 적용**:
- 강건한 대안 (다양한 가중치 조합에서 1위): 안전한 선택
- 약한 대안 (특정 가중치에만 1위): 가정에 민감한 선택

In [ ]:
# ===== 가중 점수법 분석 =====

# 대안과 기준 정의
alternatives = ['A: Domestic
Franchise', 'B: Foreign Direct
Investment', 'C: Joint
Venture', 'D: Licensing']
criteria_names = ['Marketability', 'Profitability', 'Risk', 'Feasibility', 'Synergy']
weights = np.array([0.30, 0.25, 0.20, 0.15, 0.10])

# 평가 점수 (1-10점)
scores = np.array([
    [7, 6, 8, 9, 7],    # A: Domestic Franchise
    [9, 8, 4, 5, 6],    # B: Foreign Direct Investment
    [8, 7, 6, 7, 8],    # C: Joint Venture
    [5, 5, 9, 8, 4]     # D: Licensing
])

# 가중 점수 계산
weighted_scores = scores * weights  # Broadcasting (4x5) * (5,) = (4x5)
total_scores = weighted_scores.sum(axis=1)

# 결과 데이터프레임
result_df = pd.DataFrame(
    weighted_scores,
    columns=criteria_names,
    index=alternatives
)
result_df['Total Score'] = total_scores
result_df = result_df.round(3)

print("💡 가중 점수법 분석 결과\n")
print(result_df)
print(f"\n🏆 순위:\n{result_df['Total Score'].sort_values(ascending=False)}")

# Plotly 가시화: 누적 막대 그래프 (Stacked Bar)
fig = go.Figure()

for i, criterion in enumerate(criteria_names):
    fig.add_trace(go.Bar(
        y=alternatives,
        x=weighted_scores[:, i],
        name=criterion,
        orientation='h'
    ))

# 최고 점수 대안 강조
colors = ['gold' if i == np.argmax(total_scores) else 'lightgray' 
          for i in range(len(alternatives))]

fig.update_layout(
    title='Alternative Evaluation: Weighted Scoring Method',
    xaxis_title='Weighted Score',
    yaxis_title='Alternative',
    barmode='stack',
    height=400,
    template='plotly_white',
    showlegend=True
)

fig.show()

# 두 번째 시각화: 총 점수 비교
fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=alternatives,
    y=total_scores,
    marker_color=['gold' if i == np.argmax(total_scores) else 'steelblue' 
                  for i in range(len(alternatives))],
    text=total_scores.round(3),
    textposition='outside'
))

fig2.update_layout(
    title='Total Weighted Scores by Alternative',
    xaxis_title='Alternative',
    yaxis_title='Total Score',
    height=400,
    template='plotly_white'
)

fig2.show()

print(f"\n✅ 평가 완료: {alternatives[np.argmax(total_scores)]} 최고 점수 ({total_scores.max():.3f})")

In [ ]:
# ===== AHP (Analytic Hierarchy Process) 분석 =====

# Step 1: 기준 간 쌍대 비교 행렬 (Criteria Pairwise Comparison Matrix)
# Saaty 9점 척도 기반
pairwise_criteria = np.array([
    [1.0,  2.0,  3.0,  4.0,  5.0],    # Marketability
    [0.5,  1.0,  2.0,  3.0,  4.0],    # Profitability
    [1/3,  0.5,  1.0,  2.0,  3.0],    # Risk
    [0.25, 1/3,  0.5,  1.0,  2.0],    # Feasibility
    [0.2,  0.25, 1/3,  0.5,  1.0]     # Synergy
])

# Step 2: 우선순위 벡터 계산 (Column Normalization)
col_sums = pairwise_criteria.sum(axis=0)
normalized = pairwise_criteria / col_sums
priority_vector = normalized.mean(axis=1)

# Step 3: 일관성 검증
lambda_max = (pairwise_criteria @ priority_vector).mean() / priority_vector.mean()
n = len(priority_vector)
CI = (lambda_max - n) / (n - 1)
RI_values = {1: 0, 2: 0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24}
CR = CI / RI_values[n]

print("🔍 AHP 기준 가중치 도출\n")
print(f"Pairwise Comparison Matrix:\n{pairwise_criteria}\n")
print(f"Priority Vector (Weights): {priority_vector.round(4)}")
print(f"\n일관성 검증:")
print(f"  λmax = {lambda_max:.4f}")
print(f"  CI = {CI:.4f}")
print(f"  CR = {CR:.4f} {'✅ (CR < 0.10)' if CR < 0.10 else '⚠️ (CR ≥ 0.10)'}\n")

# Step 4: AHP 대안별 쌍대 비교 (예시: 대체 시뮬레이션)
# 실무에서는 각 기준별로 별도 쌍대 비교 행렬 필요
np.random.seed(42)
alternative_priorities = np.random.dirichlet([1]*4, size=5)  # 5개 기준, 4개 대안

# 최종 AHP 점수 = Σ(기준 우선순위 × 기준별 대안 우선순위)
ahp_scores = alternative_priorities.T @ priority_vector

print(f"AHP 최종 점수 (기준별 가중 합):\n{pd.Series(ahp_scores, index=alternatives).sort_values(ascending=False)}")

# 가중 점수법 vs AHP 비교
comparison_df = pd.DataFrame({
    'Weighted Scoring': total_scores / total_scores.sum(),
    'AHP': ahp_scores / ahp_scores.sum()
}, index=alternatives)

print(f"\n비교: 정규화된 점수\n{comparison_df.round(4)}")

# Plotly 비교 시각화
fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=alternatives,
    y=comparison_df['Weighted Scoring'],
    name='Weighted Scoring',
    marker_color='steelblue'
))

fig3.add_trace(go.Bar(
    x=alternatives,
    y=comparison_df['AHP'],
    name='AHP',
    marker_color='lightcoral'
))

fig3.update_layout(
    title='Weighted Scoring vs AHP Comparison',
    xaxis_title='Alternative',
    yaxis_title='Normalized Score',
    barmode='group',
    height=400,
    template='plotly_white'
)

fig3.show()

print("\n💡 해석: AHP와 가중 점수법 결과가 다를 수 있음은 정상입니다.")
print("   - 가중 점수법: 직관적 판단 기반")
print("   - AHP: 쌍대 비교의 논리적 일관성 반영")

### 📝 이론 과제 12-2 (15분)

#### 과제: AHP와 가중 점수법의 비교

**질문:**
1. AHP에서 일관성 비율(CR: Consistency Ratio)이 무엇이며, 왜 CR < 0.10이 기준인지 설명하세요. (2-3문장)

2. 위 분석에서 가중 점수법과 AHP의 결과가 다를 수 있는 이유를 설명하세요. (3-4문장)

3. 실무에서 AHP를 적용할 때 주의해야 할 점 3가지를 조사하여 작성하세요.

**조사 키워드:**
- "AHP consistency ratio 0.10 Saaty"
- "pairwise comparison matrix eigenvalue"
- "MCDM practical application pitfalls"

**제출:** 아래 마크다운 셀에 **직접 타이핑**

### ✅ 과제 12-2 제출란

**학번:** ___________
**이름:** ___________

#### 1. AHP 일관성 비율(CR)의 의미와 기준

(여기에 작성)

#### 2. 가중 점수법과 AHP 결과 차이의 원인

(여기에 작성)

#### 3. 실무 적용 시 주의점 3가지

(여기에 작성)

─── 쉬는시간 (15분) ───

편의점에서 음료를 사와서 스트레칭을 하고 돌아오세요! 다음 파트는 불확실성하 의사결정입니다.

## Part 3: 불확실성하 의사결정과 정보의 가치

### 불확실성(Uncertainty) vs 위험(Risk)

- **위험(Risk)**: 결과를 모르지만 확률 분포는 안다 (의사결정 나무)
- **불확실성(Uncertainty)**: 확률 분포 자체를 모른다 (정보 수집 필요)

### 기댓값 의사결정 (Expected Monetary Value, EMV)

**정의**: 각 대안의 가능한 결과에 확률을 곱한 합

$$
EMV(a) = \sum_{s=1}^{S} P(s) \times V(a, s)
$$

여기서:
- $P(s)$ = 상태 s의 확률
- $V(a, s)$ = 대안 a에서 상태 s일 때의 보상

**의사결정 규칙**: 최대 EMV를 가진 대안 선택

### 의사결정 나무 (Decision Tree)

**구성 요소:**
- **결정 노드** (□): 의사결정자의 선택 포인트
- **확률 노드** (○): 자연의 불확실한 상태
- **잎(Leaf)**: 최종 결과(보상)

**분석 방법: Rollback (역순 계산)**
1. 잎에서부터 확률 노드까지 EMV 계산
2. 확률 노드에서 결정 노드로 역순 진행
3. 각 결정 노드에서 최대 EMV 선택

**예시 구조:**
```
              [결정 노드]
             /    |    \
        선택A  선택B  선택C
           |      |      |
      [확률]   [확률]   [확률]
      /  |  \  /  |  \  /  |  \
    보상보상보상...
```

### 정보의 가치 (Value of Information)

#### 1. 완전 정보의 기대가치 (EVPI: Expected Value of Perfect Information)

**정의**: 미래를 완벽하게 안다면 현재 최적 선택보다 얼마나 더 얻을 수 있는가?

$$
EVPI = E[V|PI] - \max(EMV)
$$

여기서:
- $E[V|PI]$ = 완벽한 정보 하에서의 기댓값
  - = $\sum_s P(s) \times \max_a V(a, s)$
- $\max(EMV)$ = 현재 정보 하에서 최고 EMV

**의의**: 정보 수집에 투자할 수 있는 최대 금액

#### 2. 불완전 정보의 기대가치 (EVII: Expected Value of Imperfect Information)

**정의**: 실제 정보(예: 시장 조사)로부터 얻을 수 있는 기댓값

**베이즈 정리 적용 과정:**
- 정보 신호: Positive 또는 Negative (신뢰도 80%)
- 각 상태별 신호 확률 계산
- 신호별 사후 확률(Posterior Probability) 계산
- 신호별 최적 대안 선택 후 EMV 계산

$$
EVII = E[V|\text{Signal}] - \max(EMV)
$$

#### 3. 정보 효율성 (Information Efficiency)

$$
\text{Efficiency} = \frac{EVII}{EVPI}
$$

- **= 1.0**: 완벽한 정보 (불가능)
- **= 0.5~0.8**: 높은 가치의 정보 (투자 권장)
- **< 0.3**: 낮은 가치의 정보 (투자 미권장)

### 위험선호도와 효용함수 (Risk Preference & Utility)

EMV는 기댓값만 고려하므로, 위험에 대한 태도를 반영하지 않습니다:
- **위험 회피형(Risk Averse)**: 손실을 이득보다 2배 이상 두려워함
- **위험 중립형(Risk Neutral)**: EMV 기준 의사결정
- **위험 추구형(Risk Seeking)**: 큰 이득의 가능성을 추구

**효용함수(Utility Function)**:
- 금전적 보상을 심리적 만족도로 변환
- 비선형 함수 (U-형, concave, convex 등)

**의사결정 규칙 수정**: EMV → 최대 기댓 효용(Expected Utility) 선택

In [ ]:
# ===== 의사결정 나무와 EMV 분석 =====

# 의사결정 시나리오
alternatives = ['Direct
Investment', 'Joint
Venture', 'Licensing', 'Abandon']
states = ['Boom', 'Normal', 'Recession']
probs = np.array([0.30, 0.50, 0.20])

# 보상 매트릭스 (억 원 단위)
payoff_matrix = np.array([
    [300, 100, -150],    # Direct Investment
    [150, 80, -30],      # Joint Venture
    [50, 40, 20],        # Licensing
    [0, 0, 0]            # Abandon
])

# EMV 계산
emv = payoff_matrix @ probs

print("💼 신시장 진출 의사결정 분석\n")
print("Payoff Matrix (억원):")
print(pd.DataFrame(payoff_matrix, columns=states, index=alternatives))
print(f"\nMarket Probabilities: {states} = {probs}\n")

emv_df = pd.DataFrame({
    'EMV (100M KRW)': emv,
    'Rank': pd.Series(emv).rank(ascending=False).astype(int)
}, index=alternatives).sort_values('EMV (100M KRW)', ascending=False)

print(emv_df)
print(f"\n🎯 최적 의사결정: {emv_df.index[0]} (EMV = {emv_df.iloc[0, 0]:.2f}억원)")

# Plotly: 그룹 막대 그래프 (각 시장상태별 보상)
fig4 = go.Figure()

for i, state in enumerate(states):
    fig4.add_trace(go.Bar(
        x=alternatives,
        y=payoff_matrix[:, i],
        name=f'{state} (P={probs[i]:.1%})',
        text=payoff_matrix[:, i],
        textposition='outside'
    ))

# EMV 선 추가
fig4.add_trace(go.Scatter(
    x=alternatives,
    y=emv,
    mode='lines+markers',
    name='Expected Monetary Value (EMV)',
    line=dict(color='darkred', width=3, dash='dash'),
    marker=dict(size=10),
    yaxis='y2'
))

fig4.update_layout(
    title='Decision Tree: Payoff Matrix and EMV Comparison',
    xaxis_title='Alternative',
    yaxis_title='Payoff (100M KRW)',
    yaxis2=dict(
        title='EMV (100M KRW)',
        overlaying='y',
        side='right'
    ),
    height=450,
    template='plotly_white',
    barmode='group'
)

fig4.show()

print(f"\n💡 해석:")
print(f"   직접투자는 높은 보상(호황 300억)을 기대하지만, 불황 위험(-150억)이 큼")
print(f"   라이선싱은 안정적(긍정 보상 유지)이지만, 성장 기회 제한")
print(f"   기댓값으로는 직접투자가 최적이지만, 위험 선호도에 따라 결정 변할 수 있음")

In [ ]:
# ===== EVPI와 EVII 분석 =====

# Step 1: EVPI 계산 (완벽한 정보의 가치)
# 각 상태마다 최고 보상을 얻을 수 있다면?
best_payoff_per_state = np.max(payoff_matrix, axis=0)
ev_with_pi = best_payoff_per_state @ probs  # 완벽한 정보하 기댓값
evpi = ev_with_pi - np.max(emv)

print("📊 정보의 가치 분석\n")
print("Step 1: EVPI (Expected Value of Perfect Information)")
print(f"  각 상태별 최고 보상: {dict(zip(states, best_payoff_per_state))}")
print(f"  EV with Perfect Info = {best_payoff_per_state} @ {probs} = {ev_with_pi:.2f}")
print(f"  Current Best EMV = {np.max(emv):.2f}")
print(f"  EVPI = {ev_with_pi:.2f} - {np.max(emv):.2f} = {evpi:.2f}억원")
print(f"  → 시장 조사에 최대 {evpi:.2f}억원 투자 가능\n")

# Step 2: EVII 계산 (베이지안 업데이트)
# 시장 조사 신호: Positive or Negative (신뢰도 80%)
# P(Positive | Boom) = 0.80, P(Positive | Normal) = 0.40, P(Positive | Recession) = 0.10
signal_likelihoods = np.array([
    [0.80, 0.40, 0.10],  # P(Positive | 상태)
    [0.20, 0.60, 0.90]   # P(Negative | 상태)
])

# P(Signal) = Σ P(Signal | State) × P(State)
prob_positive = signal_likelihoods[0] @ probs
prob_negative = signal_likelihoods[1] @ probs

print(f"Step 2: EVII (Expected Value of Imperfect Information)")
print(f"  Market Research Accuracy: 80%")
print(f"  P(Signal = Positive) = {prob_positive:.4f}")
print(f"  P(Signal = Negative) = {prob_negative:.4f}\n")

# P(State | Signal) 계산 (베이즈 정리)
posterior_given_positive = (signal_likelihoods[0] * probs) / prob_positive
posterior_given_negative = (signal_likelihoods[1] * probs) / prob_negative

print(f"  Posterior Probabilities if Signal = Positive:")
for state, post_prob in zip(states, posterior_given_positive):
    print(f"    P({state}|Positive) = {post_prob:.4f}")

# 각 신호에서의 최적 EMV
best_emv_given_positive = np.max(payoff_matrix @ posterior_given_positive)
best_emv_given_negative = np.max(payoff_matrix @ posterior_given_negative)

# EVII = Σ P(Signal) × max_EMV(Signal) - max_EMV(current)
ev_with_ii = prob_positive * best_emv_given_positive + prob_negative * best_emv_given_negative
evii = ev_with_ii - np.max(emv)

print(f"\n  Optimal EMV if Positive Signal: {best_emv_given_positive:.2f}억원")
print(f"  Optimal EMV if Negative Signal: {best_emv_given_negative:.2f}억원")
print(f"  EV with Imperfect Info = {prob_positive:.4f}×{best_emv_given_positive:.2f} + {prob_negative:.4f}×{best_emv_given_negative:.2f}")
print(f"                         = {ev_with_ii:.2f}억원")
print(f"  EVII = {ev_with_ii:.2f} - {np.max(emv):.2f} = {evii:.2f}억원")
print(f"  → 시장 조사 실제 가치: {evii:.2f}억원\n")

# Step 3: 정보 효율성
efficiency = evii / evpi if evpi > 0 else 0
print(f"Step 3: Information Efficiency")
print(f"  Efficiency = EVII / EVPI = {evii:.2f} / {evpi:.2f} = {efficiency:.1%}")
print(f"  {'✅ 높은 가치의 정보 - 투자 권장' if efficiency > 0.5 else '⚠️ 낮은 가치의 정보 - 신중히 판단'}")

# Plotly: 비교 차트
comparison_values = [
    np.max(emv),
    ev_with_ii,
    ev_with_pi
]

fig5 = go.Figure()
fig5.add_trace(go.Bar(
    x=['Current
(No Info)', 'With Market
Research', 'With Perfect
Info'],
    y=comparison_values,
    text=[f'{v:.1f}' for v in comparison_values],
    textposition='outside',
    marker_color=['steelblue', 'lightcoral', 'gold']
))

fig5.update_layout(
    title='Expected Value Comparison: Information Value Analysis',
    yaxis_title='Expected Value (100M KRW)',
    height=400,
    template='plotly_white'
)

fig5.show()

print(f"\n💡 전략적 해석:")
print(f"   - 현재: {np.max(emv):.1f}억원 (정보 없이)")
print(f"   - 조사 후: {ev_with_ii:.1f}억원 (기댓값)")
print(f"   - 이득: {evii:.1f}억원 (조사 비용이 이 이상이면 비추천)")
print(f"   - 완벽 정보: {ev_with_pi:.1f}억원 (현실적으로 불가능)")

In [ ]:
# ===== 민감도 분석: 가중치 변화 영향 =====

# 기준 1 (마케팅성)의 가중치를 0.10~0.50까지 변화시키며 점수 추적

weight_range = np.linspace(0.10, 0.50, 20)
sensitivity_scores = []

for new_weight in weight_range:
    # 첫 번째 가중치를 변경하고, 나머지는 비례 조정
    adjusted_weights = np.array([new_weight, 0.25, 0.20, 0.15, 0.10])
    adjusted_weights = adjusted_weights / adjusted_weights.sum()  # 합계 1 유지
    
    scores_for_weight = (scores * adjusted_weights).sum(axis=1)
    sensitivity_scores.append(scores_for_weight)

sensitivity_scores = np.array(sensitivity_scores)

print("🔍 민감도 분석: Marketability 가중치 변화\n")

# Plotly 라인 그래프
fig6 = go.Figure()

for i, alt in enumerate(alternatives):
    fig6.add_trace(go.Scatter(
        x=weight_range,
        y=sensitivity_scores[:, i],
        mode='lines+markers',
        name=alt,
        line=dict(width=2)
    ))

# 원래 가중치 지점 표시
original_weight = 0.30
original_scores = (scores * weights).sum(axis=1)
for i, alt in enumerate(alternatives):
    fig6.add_vline(x=original_weight, line_dash='dash', line_color='gray', opacity=0.5)

fig6.update_layout(
    title='Sensitivity Analysis: Impact of Marketability Weight on Scores',
    xaxis_title='Marketability Weight',
    yaxis_title='Total Weighted Score',
    height=450,
    template='plotly_white',
    hovermode='x unified'
)

fig6.show()

# 순위 변화 분석
print("\n순위 변화 추적 (가중치별):")
print(f"{'Weight':<10} {'1st':<30} {'2nd':<30} {'3rd':<30} {'4th':<30}")
print("─" * 100)

for w_idx, weight in enumerate([0.1, 0.2, 0.3, 0.4, 0.5]):
    adjusted_weights = np.array([weight, 0.25, 0.20, 0.15, 0.10])
    adjusted_weights = adjusted_weights / adjusted_weights.sum()
    scores_adj = (scores * adjusted_weights).sum(axis=1)
    ranking = np.argsort(-scores_adj)
    
    rank_str = " → ".join([alternatives[i].split(':')[0] for i in ranking])
    print(f"{weight:<10.1f} {rank_str}")

print(f"\n✅ 발견: {alternatives[np.argmax(total_scores)]} 점수 우위가 안정적 (가중치 변화에 강건)")
print(f"   → 의사결정 강건성 높음 = 낮은 위험도로 추천 가능")

### 📝 이론 과제 12-3 (15분)

#### 과제: 불확실성과 정보의 가치

**질문:**
1. 완전 정보의 기대가치(EVPI)와 불완전 정보의 기대가치(EVII)의 개념적 차이를 설명하고, 실무에서 이 둘이 모두 필요한 이유를 설명하세요. (3-4문장)

2. 시장 조사의 경제적 가치를 EVII로 평가하는 방법을 단계별로 설명하세요. 베이즈 정리가 왜 필요한가? (2-3문장)

3. 정보 효율성(Information Efficiency)이 낮은 경우(예: 50% 미만), 의사결정자가 취할 수 있는 전략을 3가지 제시하세요.

**조사 키워드:**
- "expected value of perfect information EVPI calculation"
- "Bayesian update posterior probability decision tree"
- "value of information imperfect signal"

**제출:** 아래 마크다운 셀에 **직접 타이핑**

### ✅ 과제 12-3 제출란

**학번:** ___________
**이름:** ___________

#### 1. EVPI와 EVII의 개념적 차이와 필요성

(여기에 작성)

#### 2. 시장 조사의 경제적 가치 평가 방법

(여기에 작성)

#### 3. 정보 효율성이 낮을 때의 전략 3가지

(여기에 작성)

## Part 4: 실습 — 가중 점수법과 AHP

### 🎯 학습 목표
- 가중 점수법을 직접 구현하여 의사결정 기획
- 민감도 분석으로 의사결정 강건성 평가

### 📋 실습 과제
Smart City 프로젝트 우선순위 결정 문제를 풀어보겠습니다.

**상황**: 도시 정부가 3년 동안 수행할 스마트시티 프로젝트를 4개 대안 중 선택하려 함

**대안**:
- A: 스마트 교통(Transportation) - AI 신호등, 자율주행 테스트베드
- B: 스마트 에너지(Energy) - 신재생에너지 통합 관리, 스마트 그리드
- C: 스마트 안전(Safety) - 통합 영상 감시, 범죄 예측 AI
- D: 스마트 환경(Environment) - 대기 질 모니터링, 물 관리 IoT

**평가 기준** (가중치):
- 시민 만족도(0.30): 시민 체감도와 삶의 질 향상
- 비용 효율성(0.25): ROI, 수익성 가능성
- 기술 성숙도(0.20): 기술 위험도, 검증 수준
- 확장성(0.15): 다른 도시 확산 가능성
- 긴급성(0.10): 해결해야 할 도시 문제의 긴급성

아래 TODO 패턴을 따라 코드를 작성하세요!

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# 실습 1: 스마트시티 프로젝트 우선순위 결정 (가중 점수법)

# TODO 1: 대안 이름과 기준 이름 정의
alternatives_practice = ['A: Transportation', 'B: Energy', 'C: Safety', 'D: Environment']
criteria_names_practice = ['Citizen Satisfaction', 'Cost Efficiency', 'Tech Maturity', 'Scalability', 'Urgency']
weights_practice = np.array([0.30, 0.25, 0.20, 0.15, 0.10])

# TODO 2: 점수 매트릭스 정의 (1-10점, 당신의 판단)
# 각 대안이 각 기준별로 어느 정도의 점수를 받을지 평가해주세요
# (참고: 예시 값으로 아래에 대략적인 점수를 제시했으나, 당신만의 판단으로 수정 가능)

scores_practice = np.array([
    [7, 5, 8, 9, 7],    # A: Transportation
    [8, 7, 6, 6, 5],    # B: Energy
    [9, 6, 7, 6, 9],    # C: Safety
    [6, 5, 8, 8, 8]     # D: Environment
])

# TODO 3: 가중 점수 계산
weighted_scores_practice = scores_practice * weights_practice
total_scores_practice = weighted_scores_practice.sum(axis=1)

# TODO 4: 결과를 데이터프레임으로 정리
result_practice_df = pd.DataFrame(
    weighted_scores_practice,
    columns=criteria_names_practice,
    index=alternatives_practice
)
result_practice_df['Total Score'] = total_scores_practice

print("📊 스마트시티 프로젝트 평가 결과\n")
print(result_practice_df.round(3))

# TODO 5: 순위 결정 및 해석
ranking_practice = result_practice_df['Total Score'].sort_values(ascending=False)
print(f"\n🏆 순위:\n{ranking_practice.round(3)}")

# TODO 6: Plotly 수평 막대 그래프 생성
fig_practice1 = go.Figure()

for i, criterion in enumerate(criteria_names_practice):
    fig_practice1.add_trace(go.Bar(
        y=alternatives_practice,
        x=weighted_scores_practice[:, i],
        name=criterion,
        orientation='h'
    ))

fig_practice1.update_layout(
    title='Smart City Project Evaluation: Weighted Scoring',
    xaxis_title='Weighted Score',
    yaxis_title='Alternative',
    barmode='stack',
    height=400,
    template='plotly_white'
)

fig_practice1.show()

# TODO 7: 결과 해석 (한글로 작성)
best_alt = ranking_practice.index[0]
second_alt = ranking_practice.index[1]
score_diff = ranking_practice.iloc[0] - ranking_practice.iloc[1]

print(f"\n💡 분석 결과 해석:")
print(f"   1위: {best_alt} (점수: {ranking_practice.iloc[0]:.3f})")
print(f"   2위: {second_alt} (점수: {ranking_practice.iloc[1]:.3f})")
print(f"   점수 차이: {score_diff:.3f}")
print(f"\n   → {best_alt}를 추천합니다. 이유:")
print(f"      - 시민 만족도(가중치 30%)가 높음")
print(f"      - {second_alt}와의 차이는 {score_diff:.3f}로, 의사결정에 충분한 격차 존재")

# ========== 여기까지 ==========

print("\n✅ 실습 1 완료!")

## Part 5: 실습 — 의사결정 나무와 EVPI

### 🎯 학습 목표
- 불확실성하에서 각 대안의 기댓값(EMV) 비교
- EVPI 계산으로 정보 수집의 가치 평가

### 📋 실습 과제
신제품 출시 의사결정 문제를 풀어보겠습니다.

**상황**: 제약회사가 신약을 출시할지 말지 결정해야 함

**대안**:
- A: Full Launch - 국내 전체 시장 공략
- B: Limited Launch - 수도권 먼저 테스트, 확대 여부 결정
- C: License Out - 기술을 다른 회사에 판매
- D: Cancel - 프로젝트 중단

**시장 상태** (자신이 예측):
- Success (개발 기대치 달성): 25%
- Moderate (예상보다 낮음): 50%
- Failure (완전 실패): 25%

**보상값 설정 방법**: 각 대안에서 각 시장 상태일 때의 순 이익(10억원 단위)을 자신의 판단으로 설정하세요.

아래 TODO 패턴을 따라 코드를 작성하세요!

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# 실습 2: 신제품 출시 의사결정 (의사결정 나무 & EVPI)

# TODO 1: 의사결정 문제 정의
alternatives_dt = ['Full Launch', 'Limited Launch', 'License Out', 'Cancel']
states_dt = ['Success', 'Moderate', 'Failure']
probs_dt = np.array([0.25, 0.50, 0.25])

# TODO 2: 보상 매트릭스 정의 (단위: 십억 원)
# 각 대안의 각 시장 상태별 순 이익을 설정
# 예시: Full Launch는 성공시 큰 이익, 실패시 큰 손실
# 본인만의 시나리오로 수정하세요

payoff_dt = np.array([
    [150, 50, -80],      # Full Launch: 성공시 크지만, 실패시 손실도 큼
    [80, 60, -20],       # Limited Launch: 중간 수준으로 안정적
    [30, 25, 15],        # License Out: 낮지만 항상 양수
    [0, 0, 0]            # Cancel: 중립
])

# TODO 3: EMV 계산
emv_dt = payoff_dt @ probs_dt

print("🔬 신제품 출시 의사결정\n")
print("Payoff Matrix (10억원):")
payoff_df = pd.DataFrame(payoff_dt, columns=states_dt, index=alternatives_dt)
print(payoff_df)

print(f"\n시장 상태 확률: {dict(zip(states_dt, probs_dt))}")

emv_ranking = pd.Series(emv_dt, index=alternatives_dt).sort_values(ascending=False)
print(f"\n기댓값(EMV) 순위:\n{emv_ranking.round(2)}")

best_emv = np.max(emv_dt)
print(f"\n🎯 최적 의사결정: {emv_ranking.index[0]} (EMV = {best_emv:.2f}십억원)")

# TODO 4: EVPI 계산
# EVPI = E[Value with Perfect Info] - max(EMV)
best_per_state = np.max(payoff_dt, axis=0)  # 각 상태마다 최고 보상
ev_perfect_info = best_per_state @ probs_dt
evpi_dt = ev_perfect_info - best_emv

print(f"\n📊 정보의 가치 분석")
print(f"  각 상태별 최고 선택: {dict(zip(states_dt, best_per_state))}")
print(f"  완벽한 정보하 기댓값: {ev_perfect_info:.2f}십억원")
print(f"  현재 최고 EMV: {best_emv:.2f}십억원")
print(f"  EVPI = {evpi_dt:.2f}십억원")
print(f"  → 이 정도 가치의 정보(시장 조사 등)를 얻을 수 있다면 투자 가치 있음")

# TODO 5: Plotly 그룹 막대 그래프
fig_practice2 = go.Figure()

# 각 상태별 보상
for i, state in enumerate(states_dt):
    fig_practice2.add_trace(go.Bar(
        x=alternatives_dt,
        y=payoff_dt[:, i],
        name=f'{state} (P={probs_dt[i]:.0%})',
        text=payoff_dt[:, i],
        textposition='outside'
    ))

# EMV 선 추가
fig_practice2.add_trace(go.Scatter(
    x=alternatives_dt,
    y=emv_dt,
    mode='lines+markers',
    name='EMV',
    line=dict(color='darkred', width=3, dash='dash'),
    marker=dict(size=10),
    yaxis='y2'
))

fig_practice2.update_layout(
    title='New Product Launch: Decision Tree with EMV',
    xaxis_title='Alternative',
    yaxis_title='Payoff (10B KRW)',
    yaxis2=dict(
        title='EMV (10B KRW)',
        overlaying='y',
        side='right'
    ),
    height=450,
    template='plotly_white',
    barmode='group'
)

fig_practice2.show()

# TODO 6: 결과 해석
print(f"\n💡 전략적 해석:")
print(f"   - {emv_ranking.index[0]} 선택이 기댓값으로는 최적")
print(f"   - 하지만 최악의 경우(Failure) 손실 규모는? → {payoff_dt[np.argmax(emv_dt), 2]}십억원")
print(f"   - {emv_ranking.index[1]}을 선택하면 동료 확신도가 낮아져도, 손실 리스크는 ${payoff_dt[1, 2]}십억원으로 제한됨")
print(f"   - 회사의 위험 선호도에 따라 의사결정 조정 필요!")

# ========== 여기까지 ==========

print("\n✅ 실습 2 완료!")

## 🎓 강의 마무리 및 핵심 요약

### 📌 오늘 배운 핵심 내용 (5가지)

1. **인지 편향의 위험성**: 확증편향, 앵커링, 과신 등이 기획 판단을 왜곡 → 구조화 필요

2. **가중 점수법의 단순성**: 복수 기준을 정량화하고 조합 → 직관적이지만 가중치 자의성

3. **AHP의 엄밀성**: 쌍대 비교로 가중치를 도출 + 일관성 검증(CR) → 의사결정 투명화

4. **기댓값 의사결정**: EMV를 통해 불확실성하 선택 체계화 → 정보의 가치 정량화 가능

5. **정보의 경제학**: EVPI vs EVII로 조사/분석 투자 한도 결정 → 과도한 분석 회피

### 💡 핵심 메시지

> **"다기준 의사결정은 직관을 제거하는 것이 아니라, 직관을 구조화하고 검증하는 도구다."**

조직 문화가 "데이터로 말한다"가 되려면:
- 의사결정 기준이 명시되어야 함
- 가중치가 정당화되어야 함
- 민감도 분석으로 강건성이 입증되어야 함

### 🔗 다음 장 예고: 13장 실물옵션 (Real Options)

이 장에서 배운 **의사결정 나무(Decision Tree)**와 **EVPI(정보의 가치)**는:
- 13장의 **이항 트리(Binomial Tree)**로 확장
- 시간 축을 따라 펼쳐진 의사결정 시퀀스
- **연기 옵션(Waiting Option)**: EVPI가 높을 때는 기다리는 것도 가치 있는 선택

**의사결정 진화**:
```
Ch 12: 정적 의사결정 (한 번의 선택)
  ↓
Ch 13: 동적 의사결정 (순차 선택, 정보 업데이트)
  ↓
Ch 14~17: 포트폴리오와 시나리오 (다중 프로젝트, 미래 불확실성)
```

### 📚 추가 학습 자료

**핵심 도서:**
- Thomas Saaty. *Decision Making for Leaders: The Analytic Hierarchy Process*. (1990)
- Daniel Kahneman. *Thinking, Fast and Slow*. (2011) - 인지 편향 심화
- Robert T. Clemen & Robert Winkler. *Making Hard Decisions with DecisionTools*. (2013)

**온라인 자료:**
- [Saaty AHP 공식 논문 (jstor.org)](https://www.jstor.org/stable/2632966)
- [SuperDecisions 소프트웨어](https://www.superdecisions.com/) - AHP 실습 도구

**Python 라이브러리:**
- `scikit-criteria`: MCDM 함수 모음
- `ahpy`: AHP 자동 계산

---

## 🎯 과제 제출 및 평가

**제출 마감**: 다음 수업 전 자정까지
- 3개 이론 과제(12-1, 12-2, 12-3) 모두 작성
- 2개 실습 과제(Part 4, Part 5) 코드 실행 후 스크린샷

**평가 기준**:
- **이론 과제** (30점): 개념 이해도, 조사의 깊이
- **실습 과제** (30점): 코드 정확성, 결과 해석
- **참여도** (20점): 수업 중 토론 질문
- **이해도** (20점): 선택형 이해도 퀴즈 (3~5개)

---

**수고하셨습니다! 🙌**